In [1]:
cd ..

/Users/pardhu/Developer/TMF Classfier


In [2]:
import pandas as pd
import re
from transformers import AutoTokenizer
from tqdm import tqdm

[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


In [3]:
df = pd.read_csv("raw_extracted_dataset.csv")

print(df.shape)
df.head()

(53, 4)


,file_name,class,text,num_chars
0,Prot_008.pdf,protocol,Clinical Study Protocol \nDrug Substance Durva...,271411
1,Prot_009.pdf,protocol,Official Protocol Title: \nNCT number: \nNCT03...,358803
2,Prot_007.pdf,protocol,Official Protocol Title:\nNCT number:\nDocumen...,187601
3,Prot_013.pdf,protocol,PF-06863135 (Elranatamab)\nProtocol C1071003\n...,342031
4,Prot_012.pdf,protocol,"This may include, but is not limited to, redac...",212928


In [4]:
def clean_text(text):

    text = str(text)

    # remove extra spaces/newlines
    text = re.sub(r"\s+", " ", text)

    # remove weird characters
    text = re.sub(r"[^\x00-\x7F]+", " ", text)

    # remove multiple spaces
    text = re.sub(r" +", " ", text)

    return text.strip()

In [5]:
df["clean_text"] = df["text"].apply(clean_text)

df[["class", "clean_text"]].head()

,class,clean_text
0,protocol,Clinical Study Protocol Drug Substance Durvalu...
1,protocol,Official Protocol Title: NCT number: NCT033022...
2,protocol,Official Protocol Title: NCT number: Document ...
3,protocol,PF-06863135 (Elranatamab) Protocol C1071003 Fi...
4,protocol,"This may include, but is not limited to, redac..."


In [6]:
MODEL_NAME = "emilyalsentzer/Bio_ClinicalBERT"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

In [7]:
def chunk_text(text,
               tokenizer,
               max_tokens=512,
               overlap=50):

    tokens = tokenizer.encode(
        text,
        add_special_tokens=False
    )

    chunks = []

    start = 0

    while start < len(tokens):

        end = start + max_tokens

        chunk_tokens = tokens[start:end]

        chunk = tokenizer.decode(
            chunk_tokens,
            skip_special_tokens=True
        )

        chunks.append(chunk)

        start += (max_tokens - overlap)

    return chunks

In [8]:
chunk_dataset = []

for _, row in tqdm(df.iterrows(),
                   total=len(df)):

    chunks = chunk_text(
        row["clean_text"],
        tokenizer
    )

    for idx, chunk in enumerate(chunks):

        chunk_dataset.append({

            "file_name": row["file_name"],

            "class": row["class"],

            "chunk_id": idx,

            "chunk_text": chunk,

            "chunk_chars": len(chunk)

        })

100%|██████████| 53/53 [00:13<00:00,  3.94it/s]


In [9]:
chunk_df = pd.DataFrame(chunk_dataset)

print(chunk_df.shape)

chunk_df.head()

(4078, 5)


,file_name,class,chunk_id,chunk_text,chunk_chars
0,Prot_008.pdf,protocol,0,clinical study protocol drug substance durvalu...,1807
1,Prot_008.pdf,protocol,1,"1. 0, 05 - jun - 2017 name of authorized perso...",1877
2,Prot_008.pdf,protocol,2,lymphocyte subsets within the tumor microenvir...,2206
3,Prot_008.pdf,protocol,3,"on days - 1 & 1 and of cycle 1, and then on da...",1983
4,Prot_008.pdf,protocol,4,2 will clinical study protocol drug substance ...,2309


In [10]:
chunk_df = pd.DataFrame(chunk_dataset)

print(chunk_df.shape)

chunk_df.head()

(4078, 5)


,file_name,class,chunk_id,chunk_text,chunk_chars
0,Prot_008.pdf,protocol,0,clinical study protocol drug substance durvalu...,1807
1,Prot_008.pdf,protocol,1,"1. 0, 05 - jun - 2017 name of authorized perso...",1877
2,Prot_008.pdf,protocol,2,lymphocyte subsets within the tumor microenvir...,2206
3,Prot_008.pdf,protocol,3,"on days - 1 & 1 and of cycle 1, and then on da...",1983
4,Prot_008.pdf,protocol,4,2 will clinical study protocol drug substance ...,2309


In [11]:
chunk_df_v1 = chunk_df[
    chunk_df["class"] != "informed_consent"
].reset_index(drop=True)

chunk_df_v1["class"].value_counts()

class
protocol                     1986
statistical_analysis_plan    1335
safety_report                 706
Name: count, dtype: int64

In [12]:
target_per_class = 700

balanced_chunk_df = (
    chunk_df_v1
    .groupby("class", group_keys=False)
    .apply(
        lambda x: x.sample(
            n=min(len(x), target_per_class),
            random_state=42
        )
    )
    .reset_index(drop=True)
)

balanced_chunk_df["class"].value_counts()

/var/folders/fn/knzvwqfx529d_ynxfzrct8f80000gn/T/ipykernel_15575/251611839.py:6: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(


class
protocol                     700
safety_report                700
statistical_analysis_plan    700
Name: count, dtype: int64

In [13]:
balanced_chunk_df.groupby("class").agg(
    total_chunks=("chunk_text", "count"),
    avg_chunk_chars=("chunk_chars", "mean"),
    min_chunk_chars=("chunk_chars", "min"),
    max_chunk_chars=("chunk_chars", "max")
)

,total_chunks,avg_chunk_chars,min_chunk_chars,max_chunk_chars
class,,,,
protocol,700,1913.488571,272,2966
safety_report,700,2203.084286,314,2926
statistical_analysis_plan,700,1812.327143,26,2872


In [14]:
for i in range(5):

    print("=" * 100)

    print("CLASS:",
          balanced_chunk_df.iloc[i]["class"])

    print("CHUNK ID:",
          balanced_chunk_df.iloc[i]["chunk_id"])

    print("=" * 100)

    print(
        balanced_chunk_df.iloc[i]["chunk_text"][:2000]
    )

    print("\n\n")

CLASS: protocol
CHUNK ID: 54
pd - l2 ). based on preclinical in vitro data, pembrolizumab has high affinity and potent receptor blocking activity for pd - 1. pembrolizumab has an acceptable preclinical safety profile and is in clinical development as an iv immunotherapy for advanced malignancies. keytruda ( pembrolizumab ) is indicated for the treatment of patients across a number of indications. for more details on specific indications, refer to the ib. refer to the ib / approved labeling for detailed background information on mk - 3475. 05nqkr 06dzns product : mk - 3475 23 protocol / amendment no. : 598 - 06 mk - 3475 - 598 - 06 final protocol 11 - dec - 2020 confidential pharmaceutical and therapeutic background the importance of intact immune surveillance in controlling outgrowth of neoplastic transformation has been known for decades [ 2 ]. accumulating evidence shows a correlation between tumor - infiltrating lymphocytes in cancer tissue and favorable prognosis in various maligna

In [15]:
balanced_chunk_df.to_csv(
    "bert_3class_chunk_dataset.csv",
    index=False
)

print("Saved 3-class chunk dataset successfully.")

Saved 3-class chunk dataset successfully.
